In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# 데이터 불러오기
x = pd.read_csv("/kaggle/input/data-science-london-scikit-learn/train.csv",header=None)
y = pd.read_csv("/kaggle/input/data-science-london-scikit-learn/trainLabels.csv",header=None)
test = pd.read_csv("/kaggle/input/data-science-london-scikit-learn/test.csv",header=None)

y = y.values.ravel()

In [ ]:
from sklearn.model_selection import train_test_split

x.isnull().sum() # 결측치 없음
train_x, test_x, train_y, test_y = train_test_split(x, y, test_size=0.4, random_state=4) # 데이터 분할
train_y = train_y

In [ ]:
# 로지스틱 회귀
from sklearn.linear_model import LogisticRegression 
from sklearn.metrics import accuracy_score

# 회귀 분석
modelLR = LogisticRegression()
modelLR.fit(train_x, train_y)

# 예측
predLR = modelLR.predict(test_x) 
accuracyLR = accuracy_score(test_y, predLR)
print("정확도:", accuracyLR)

In [ ]:
# 로지스틱 (변수 선택)
import sklearn.feature_selection as fs

selectLR1 = fs.SelectFromModel(modelLR)
selectLR1.fit(train_x, train_y)

selectLR2 = fs.RFE(modelLR)
selectLR2.fit(train_x, train_y)

# 변수 선택
selected_trxLR1 = selectLR1.transform(train_x)
selected_testxLR1 = selectLR1.transform(test_x)

selected_trxLR2 = selectLR2.transform(train_x)
selected_testxLR2 = selectLR2.transform(test_x)


# 회귀 분석
mLR1 = LogisticRegression().fit(selected_trxLR1, train_y)
mLR2 = LogisticRegression().fit(selected_trxLR2, train_y)

# 예측
predLR1 = mLR1.predict(selected_testxLR1) 
accuracyLR1 = accuracy_score(test_y, predLR1)
print("정확도:", accuracyLR1)

predLR2 = mLR2.predict(selected_testxLR2) 
accuracyLR2 = accuracy_score(test_y, predLR2)
print("정확도:", accuracyLR2)

In [ ]:
selectLRf = fs.SelectFromModel(modelLR)
selectLRf.fit(x, y)

selected_LRx = selectLRf.transform(x)

mLR1 = LogisticRegression().fit(selected_LRx, y)

selected_testLR = selectLRf.transform(test)
L_out = mLR1.predict(selected_testLR)

outputLR = pd.DataFrame({'ID':range(1, len(L_out)+1),'Solution':L_out})
outputLR.to_csv('outputLR.csv',index=False) # >> 0.80066

In [ ]:
# 의사결정나무
from sklearn.tree import DecisionTreeClassifier

modelDT = DecisionTreeClassifier()
mDT = DecisionTreeClassifier().fit(train_x, train_y)

# 가지치기
modelDT_pruned = DecisionTreeClassifier(max_depth=5, min_samples_split=15, min_samples_leaf=10)
mDT_pruned = modelDT_pruned.fit(train_x, train_y)

# 예측
predDT= mDT.predict(test_x)
accuracyDT = accuracy_score(test_y, predDT)
print("정확도:", accuracyDT)

predDT_p= mDT_pruned.predict(test_x)
accuracyDT_p = accuracy_score(test_y, predDT_p)
print("정확도:", accuracyDT_p)

In [ ]:
mDTf = modelDT_pruned.fit(x, y)

D_out = mDTf.predict(test)

outputDT = pd.DataFrame({'ID':range(1, len(D_out)+1),'Solution':D_out})
outputDT.to_csv('outputDT.csv',index=False) # >> 0.80256

In [ ]:
# 랜덤 포레스트
from sklearn.ensemble import RandomForestClassifier

modelRF = RandomForestClassifier()
mRF = RandomForestClassifier().fit(train_x, train_y)

# 예측
predRF= mRF.predict(test_x)
accuracyRF = accuracy_score(test_y, predRF)
print("정확도:", accuracyRF)

In [ ]:
# 최적화
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [10, 50, 100, 200, 500],
    'max_depth': [None, 5, 10, 15, 20],
    'max_features': ['sqrt']
}
grid_searchRF = GridSearchCV(estimator=modelRF, param_grid=param_grid, cv=5)
mRF2 = grid_searchRF.fit(train_x, train_y)

best_modelRF = mRF2.best_estimator_
best_paramsRF = mRF2.best_params_
print("Best Parameters:", best_paramsRF)

# 예측
predRF2 = best_modelRF.predict(test_x)
accuracyRF2 = accuracy_score(test_y, predRF2)
print("정확도:", accuracyRF2)

In [ ]:
mRFf = grid_searchRF.fit(x, y)
mRFf = mRFf.best_estimator_
RF_out = mRFf.predict(test)

outputRF = pd.DataFrame({'ID':range(1, len(RF_out)+1),'Solution':RF_out})
outputRF.to_csv('outputRF.csv',index=False) # >> 0.8679

In [ ]:
# 서포트 벡터 머신
from sklearn.svm import SVC

modelSVC = SVC()
mSVC = modelSVC.fit(train_x, train_y)

# 예측
predSVC= mSVC.predict(test_x)
accuracySVC = accuracy_score(test_y, predSVC)
print("정확도:", accuracySVC)

In [ ]:
# 최적화
param_grid = {
    'C': [1, 10, 50, 100, 200],
    'kernel': ['linear', 'poly', 'rbf', 'sigmoid']
}

grid_searchSVC = GridSearchCV(estimator=SVC(), param_grid=param_grid, cv=5)
mSVC2 = grid_searchSVC.fit(train_x, train_y)

# 최적의 모델 및 파라미터 확인
best_modelSVC = mSVC2.best_estimator_
best_paramsSVC = mSVC2.best_params_
print("Best Parameters:", best_paramsSVC)

# 최적의 모델로 예측
predSVC2 = best_modelSVC.predict(test_x)
accuracySVC2 = accuracy_score(test_y, predSVC2)
print("정확도:", accuracySVC2)

In [ ]:
mSVCf = grid_searchSVC.fit(x, y)
mSVCf = mSVCf.best_estimator_
SVC_out = mSVCf.predict(test)

outputSVC = pd.DataFrame({'ID':range(1, len(SVC_out)+1),'Solution':SVC_out})
outputSVC.to_csv('outputSVC.csv',index=False) # >> 0.9145

In [ ]:
# MLP
from sklearn.neural_network import MLPClassifier

modelMLP = MLPClassifier(max_iter=1000, early_stopping=True)
mMLP = modelMLP.fit(train_x, train_y)

# 예측
predMLP = mMLP.predict(test_x)
accuracyMLP = accuracy_score(test_y, predMLP)
print("정확도:", accuracyMLP)

In [ ]:
# 최적화
param_grid = {
    'hidden_layer_sizes': [(3,), (10,), (50,), (100,)],
    'activation': ['logistic', 'tanh', 'relu'],
    'solver': ['adam', 'sgd'],
    'alpha': [0.001],
    'learning_rate': ['constant', 'adaptive']
}

grid_searchMLP = GridSearchCV(estimator=modelMLP, param_grid=param_grid, cv=5)
mMLP2 = grid_searchMLP.fit(train_x, train_y)

best_modelMLP = mMLP2.best_estimator_
best_paramsMLP = mMLP2.best_params_
print("Best Parameters:", best_paramsMLP)

# 예측
predMLP2 = best_modelMLP.predict(test_x)
accuracyMLP2= accuracy_score(test_y, predMLP2)
print("정확도:", accuracyMLP2)

In [ ]:
mMLPf = grid_searchMLP.fit(x, y)
mMLPf = mMLPf.best_estimator_
MLP_out = mMLPf.predict(test)

outputMLP = pd.DataFrame({'ID':range(1, len(MLP_out)+1),'Solution':MLP_out})
outputMLP.to_csv('outputMLP.csv',index=False) # >> 0.80905